# 04 — Importance des features (rôle C)

Sur quoi le modèle final de B (forêt aléatoire, `scripts/train.py`) s'appuie-t-il pour reconnaître
une famille ? Ce notebook **lit** les résultats de `scripts/importance.py` (aucun long calcul ici) :

```
python -m scripts.train                               # modèle final : models/best_model.joblib
python -m scripts.hash_lookup index C:\ember2024\processed\Win32_top10   # vrais noms des colonnes hachées
python -m scripts.importance --ablation               # environ 30 min ; sans --ablation : quelques minutes
```

Toutes les mesures sont faites sur le **test** (le modèle final a vu tout le train). Métrique : F1
macro, comme B. Figures dans `results/figures/` (préfixe `C_`).

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from malware_family.dataset import find_dataset
from scripts.hash_lookup import load_index, names_for

RESULTS, FIGURES = ROOT / "results", ROOT / "results" / "figures"
if not (RESULTS / "importance_info.json").is_file():
    raise FileNotFoundError("Lancez d'abord « python -m scripts.importance » depuis la racine du projet.")
info = json.loads((RESULTS / "importance_info.json").read_text(encoding="utf-8"))
metrics = json.loads((RESULTS / "metrics.json").read_text(encoding="utf-8"))
groups = pd.read_csv(RESULTS / "importances_groupes.csv")
features = pd.read_csv(RESULTS / "importances.csv")
confusions = pd.read_csv(RESULTS / "confusions_features.csv")
families = metrics["families"]

DATASET = find_dataset(ROOT)
try:
    index = load_index(DATASET)
except FileNotFoundError as error:
    index = None
    print(error)
pd.set_option("display.max_colwidth", 80)
print(f"Modèle : {info['model_type']} ({metrics['selected_model']}), F1 macro du test "
      f"{info['baseline_f1_macro']:.4f} sur {info['n_test_rows']} lignes"
      + (" (TIRAGE : relancez sans --max-rows pour le rapport)" if info["sampled"] else ""))
print(f"Prédictions identiques à results/predictions_test.csv : {info['agreement_with_predictions_test']}")
print(f"{info['n_repeats']} permutations par groupe ou feature, graine {info['random_state']}")
print(f"Importance interne : {info.get('native_importance', 'MDI')}")

## 1. Quels groupes de features comptent le plus ?

Deux mesures, à lire ensemble :

- **Permutation** (à gauche) : on mélange toutes les colonnes d'un groupe entre les fichiers du
  test, puis on mesure la baisse du F1 macro. Grande baisse = le modèle a besoin de ce groupe.
- **Importance interne** (à droite) : le MDI pour une forêt aléatoire (part du groupe dans les
  découpages de la forêt), le gain pour LightGBM ; mesurée sur le train, normalisée à 1.
  Il favorise les features continues aux nombreuses valeurs (histogrammes, tailles), et un
  groupe redondant avec un autre peut avoir un grand MDI mais une petite baisse par permutation.

In [ ]:
display(Image(FIGURES / "C_importance_par_groupe.png"))
table = groups[["group", "n_features", "perm_f1_drop_mean", "perm_f1_drop_std", "mdi_sum"]].copy()
table["rang_permutation"] = table.perm_f1_drop_mean.rank(ascending=False).astype(int)
table["rang_mdi"] = table.mdi_sum.rank(ascending=False).astype(int)
table.round(4)

**À compléter (C)** : quels groupes dominent ? Les deux mesures sont-elles d'accord ? Un
groupe à forte MDI mais faible permutation est sans doute redondant avec un autre (par exemple
`histogram` et `byteentropy`, ou `section` et `datadirectories`). Comparer avec les pistes de
l'exploration (rapport, section 3.5).

## 2. Les features les plus importantes, avec leurs vrais noms

Les 50 meilleures features du MDI ont été permutées une par une. Pour une colonne hachée
(`…_hashNN`), plusieurs noms partagent la case : le tableau suivant les liste, avec le nombre de
fichiers du train de chaque famille qui les contiennent.

In [ ]:
display(Image(FIGURES / "C_top_features.png"))
top = features.dropna(subset=["perm_f1_drop_mean"]).head(20)
top[["feature", "group", "perm_f1_drop_mean", "perm_f1_drop_std", "mdi", "mdi_rank", "name"]].round(4)

In [ ]:
# Tous les noms derrière les colonnes hachées du top 20, et les familles qui les utilisent.
if index is not None:
    for column in top.feature[top.feature.str.contains("_hash")]:
        print(f"\n{column}")
        display(names_for(column, index).head(5)[["text", "sign", "n_samples", *families]])

**À compléter (C)** : que signifient ces features ? Par exemple une fonction importée
(allocation mémoire, réseau, registre, chiffrement), une section typique d'un packer (`UPX0`,
`.aspack`…), une entropie élevée (contenu compressé ou chiffré), l'absence de signature. Relier
aux descriptions publiques des familles (Malpedia, rapports d'éditeurs), **sans jamais
télécharger d'échantillon**.

## 3. Chaque famille a ses propres indices

Pour chaque groupe permuté (ligne), la baisse du F1 de chaque famille (colonne). Attention aux
petites familles du test (sinowal : 9 fichiers, sfone : 66, grandoreiro : 113) : leur F1 varie
beaucoup d'une permutation à l'autre.

In [ ]:
display(Image(FIGURES / "C_importance_groupes_par_famille.png"))
by_family = groups.set_index("group")[[f"f1_drop_{f}" for f in families]]
by_family.columns = families
pd.DataFrame({"groupe le plus utile": by_family.idxmax(), "baisse du F1": by_family.max().round(3),
              "F1 de référence": pd.Series(info["baseline_f1_per_family"]).round(3)})

**À compléter (C)** : quelles familles dépendent d'un seul groupe, lesquelles de plusieurs ?
Faire le lien avec les profils de la section 3.5 (grandoreiro sans imports, glupteba et berbew
sans Rich header, expiro proche d'un programme normal…).

## 3 bis. SHAP : les indices de chaque famille

Reprise de l'analyse de Kenneth. Pour au plus 200 fichiers du test de chaque famille, SHAP répartit
la décision « cette famille » entre les features : chaque fichier reçoit, pour chaque feature, une
contribution positive (la feature pousse vers la famille) ou négative (elle en éloigne). La carte
donne la part de chaque groupe dans ces contributions (en valeur absolue), famille par famille.

Différence avec la section 3 : la permutation mesure ce que le modèle **perd** sans un groupe ;
SHAP mesure sur quoi il **s'appuie** pour chaque fichier, même si un autre groupe pourrait le
remplacer. Exact et rapide pour LightGBM ; pour une forêt, il faut le paquet `shap`.

In [ ]:
if (RESULTS / "shap_groupes_par_famille.csv").is_file():
    display(Image(FIGURES / "C_shap_groupes_par_famille.png"))
    shap_top = pd.read_csv(RESULTS / "shap_top_par_famille.csv")
    shap_top["sens"] = shap_top.shap_mean.map(lambda v: "pousse vers" if v > 0 else "éloigne de")
    display(shap_top[["famille", "rang", "name", "part_pct", "sens"]].round(1))
else:
    print("Pas de SHAP : relancez « python -m scripts.importance » (avec le paquet shap pour une forêt).")

**À compléter (C)** : les indices propres à chaque famille ont-ils un sens ? Par exemple
`mfc42.dll` (bibliothèque MFC) pour cosmu, `mpr.dll` (partages réseau Windows) pour sfone : à
confirmer avec `python -m scripts.hash_lookup noms <colonne>`, plusieurs noms partageant une
case. Les familles de la section 3 et de cette section sont-elles décrites de la même façon ?

## 4. Pourquoi le modèle se trompe

Pour les paires (famille réelle → famille prédite) les plus fréquentes parmi les erreurs, on
compare, sur les 30 features les plus importantes, la moyenne des fichiers **confondus** à celle
des fichiers bien classés des deux familles. `rapprochement` proche de 1 : sur cette feature, les
fichiers confondus ressemblent à la famille prédite ; proche de 0 : à leur vraie famille.

In [ ]:
print("Paires les plus confondues :", info["confused_pairs"])
for (real, predicted), pair in confusions.groupby(["famille_reelle", "famille_predite"], sort=False):
    print(f"\n{real} classés {predicted} ({pair.n_confondus.iloc[0]} fichiers) : features qui les rapprochent")
    display(pair[pair.rapprochement >= 0.7].sort_values("rapprochement", ascending=False).head(8)[
        ["name", "moyenne_reelle_ok", "moyenne_confondus", "moyenne_predite_ok", "rapprochement"]].round(4))

**À compléter (C, avec B)** : la plupart des erreurs vont vers **wacatac**, une étiquette
générique (confiance médiane 0,13, section 2.6) : ces fichiers ressemblent-ils vraiment à
wacatac, ou l'étiquette de wacatac est-elle trop large ? Relier à la matrice de confusion de B.

## 5. Ablation : un groupe seul, ou tous sauf un

Le modèle de B réentraîné avec **un seul groupe** (ce groupe suffit-il ?) puis avec **tous les
groupes sauf un** (les autres le remplacent-ils ?). Seulement si `scripts.importance` a été lancé
avec `--ablation`.

In [ ]:
if (RESULTS / "ablation.csv").is_file():
    display(Image(FIGURES / "C_ablation.png"))
    ablation = pd.read_csv(RESULTS / "ablation.csv")
    display(ablation.pivot(index="group", columns="mode", values="test_f1_macro")
            .sort_values("seul", ascending=False).round(4))
else:
    print("Pas d'ablation : python -m scripts.importance --ablation")

**À compléter (C)** : un groupe seul suffit-il presque ? Retirer un groupe fait-il baisser
le score (groupe irremplaçable) ou non (information redondante) ? À comparer avec la
permutation : elle sous-estime l'importance des features corrélées.

## 6. À transmettre à B : un modèle réduit ?

Les groupes et les features les plus importants, pour que B teste un modèle plus petit.

In [ ]:
best_groups = list(groups.group[groups.perm_f1_drop_mean > 0.005])
best_features = list(features.dropna(subset=["perm_f1_drop_mean"]).query("perm_f1_drop_mean > 0").feature)
print(f"Groupes utiles (baisse > 0,005) : {best_groups}")
print(f"{len(best_features)} features à baisse positive : {best_features[:30]}")

## 7. Limites à discuter (rapport, section 7)

- **Hashing** : plusieurs noms partagent une colonne ; on ne sait pas lequel compte.
- **Features corrélées** : en permuter une seule laisse les autres porter l'information, donc la
  permutation sous-estime leur importance (d'où la permutation par groupe et l'ablation).
- **F1 macro** : chaque famille pèse autant ; sinowal (9 fichiers du test) pèse autant que berbew
  (10 000), et sa variation brouille la mesure.
- **Étiquettes bruitées** (ClarAVy, wacatac générique), **dérive temporelle**, **monde fermé**
  (58 % des malwares du test sont d'autres familles), plafond de 10 000 fichiers par famille.
- L'importance explique **le modèle**, pas la cause : une feature importante peut refléter un
  outil de compilation ou un packer plutôt qu'un comportement malveillant.